# **CNN-B**

A partir del estudio de búsqueda de hiperparámetros (`OPT-CNN-B`), se seleccionan los 3 trials con menor `val_loss` entre los completados, y se reentrena cada configuración bajo el nombre `CNN-B-1`, `CNN-B-2` y `CNN-B-3`. Los resultados se evalúan sobre el conjunto de test para seleccionar el modelo final de esta arquitectura para la Fase 2.

In [1]:
# Librerías
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import gc
import csv
import json
import time
import random
import subprocess
import numpy as np
import pandas as pd
import psutil
import pynvml
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score

C:\Users\Victoria\AppData\Local\Temp\ipykernel_28192\2900474774.py:13: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml


In [2]:
# Verificar GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.set_logical_device_configuration(
        gpus[0],
        [tf.config.LogicalDeviceConfiguration(memory_limit=15360)]  # Límite de 15 GB
    )
    print(f'GPU encontrada: {gpus[0].name} (límite 15GB)')
else:
    print('No se encontró GPU, corriendo en CPU')

GPU encontrada: /physical_device:GPU:0 (límite 15GB)


### **Configuración**

In [ ]:
# ── PATHS ─────────────────────────────────────────────────────────────────────
REPO_ROOT   = Path(subprocess.run(['git', 'rev-parse', '--show-toplevel'], capture_output=True, text=True).stdout.strip())
DATA_PATH   = Path.home() / 'pfc' / 'Dataset Sintetico 352x192'
MODELS_PATH = REPO_ROOT / 'Códigos' / 'Modelos' / 'Fase 1' / 'CNN-B'
MODELS_PATH.mkdir(parents=True, exist_ok=True)

In [3]:
# ── CONFIGURACIÓN GENERAL ──────────────────────────────────────────────────────
ARQUITECTURA  = 'CNN-B'
DATASET       = 'Sintético'
OPTUNA_RUN_ID = 'OPT-CNN-B'
NUM_CLASSES   = 5
CLASSES       = ['CD', 'HYP', 'MI', 'NORM', 'STTC']
CHANNELS      = 1

# Imagen
IMG_HEIGHT = 192
IMG_WIDTH  = 352
BATCH_SIZE = 32
SEED       = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED']         = str(SEED)
os.environ['TF_DETERMINISTIC_OPS']   = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
tf.config.experimental.enable_op_determinism()

# Modelo (fijos durante la búsqueda Optuna)
DENSE_UNITS   = 128
OPTIMIZER     = 'Adam'
LOSS_FUNCTION = 'sparse_categorical_crossentropy'
EPOCH_MAX     = 100

# Callbacks (mismos valores que OPT-CNN-B)
ES_PATIENCE   = 8
ES_MIN_DELTA  = 0.0001
RLR_FACTOR    = 0.5
RLR_PATIENCE  = 5
RLR_MIN_DELTA = 0.0001
RLR_MIN_LR    = 1e-7

### **Selección de los Top-3 Modelos**

Se cargan los resultados del estudio Optuna (`OPT-CNN-B.csv`) y se seleccionan los 3 trials completados (no podados) con menor `val_loss`.

In [4]:
# ── Top-3 trials de Optuna ──────────────────────────────────────────────────────
df_opt      = pd.read_csv(MODELS_PATH / f'{OPTUNA_RUN_ID}.csv', index_col='trial')
df_complete = df_opt[df_opt['state'] == 'COMPLETE'].copy()
top3        = df_complete.sort_values('value').head(3)

CONFIGS = []
for rank, (trial_num, row) in enumerate(top3.iterrows(), start=1):
    CONFIGS.append({
        'run_id':          f'CNN-B-{rank}',
        'optuna_trial':    int(trial_num),
        'optuna_val_loss': float(row['value']),
        'epochs_optuna':   int(row['epochs_trained']),
        'num_blocks':      int(row['num_blocks']),
        'base_filters':    int(row['base_filters']),
        'dropout':         float(row['dropout']),
        'learning_rate':   float(row['learning_rate']),
    })

In [7]:
for cfg in CONFIGS:
    print(f"{cfg['run_id']}  (trial Optuna #{cfg['optuna_trial']} — val_loss Optuna: {cfg['optuna_val_loss']:.4f}, {cfg['epochs_optuna']} épocas)")
    print(f"  num_blocks    : {cfg['num_blocks']}")
    print(f"  base_filters  : {cfg['base_filters']}")
    print(f"  dense_units   : {DENSE_UNITS}")
    print(f"  dropout       : {cfg['dropout']:.4f}")
    print(f"  learning_rate : {cfg['learning_rate']:.2e}")
    print(f"  optimizer     : {OPTIMIZER}")
    print()

CNN-B-1  (trial Optuna #29 — val_loss Optuna: 0.5725, 39 épocas)
  num_blocks    : 4
  base_filters  : 32
  dense_units   : 128
  dropout       : 0.2662
  learning_rate : 1.20e-04
  optimizer     : Adam

CNN-B-2  (trial Optuna #1 — val_loss Optuna: 0.5768, 45 épocas)
  num_blocks    : 4
  base_filters  : 32
  dense_units   : 128
  dropout       : 0.2796
  learning_rate : 1.43e-04
  optimizer     : Adam

CNN-B-3  (trial Optuna #28 — val_loss Optuna: 0.5827, 29 épocas)
  num_blocks    : 4
  base_filters  : 32
  dense_units   : 128
  dropout       : 0.2615
  learning_rate : 1.20e-04
  optimizer     : Adam



### **Carga de Datos**

In [8]:
# Carga de datos con tf.data
AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(directory, shuffle):
    ds = tf.keras.utils.image_dataset_from_directory(
        directory,
        image_size  = (IMG_HEIGHT, IMG_WIDTH),
        color_mode  = 'grayscale',
        class_names = CLASSES,
        batch_size  = BATCH_SIZE,
        shuffle     = shuffle,
        seed        = SEED if shuffle else None,
        label_mode  = 'int',
    )
    return (
        ds.map(lambda x, y: (x / 255.0, y), num_parallel_calls=AUTOTUNE)
          .prefetch(AUTOTUNE)
    )

# NOTA: train_data y val_data se recrean dentro del loop de entrenamiento, una vez por run_id,
# para garantizar que cada modelo empiece con el mismo estado del dataset.
test_data = make_dataset(DATA_PATH / 'test', shuffle=False)

print('Clases:', CLASSES)

Found 10258 files belonging to 5 classes.
Clases: ['CD', 'HYP', 'MI', 'NORM', 'STTC']


### **Arquitectura**

In [9]:
def build_cnn_b(num_blocks, base_filters, dense_units, dropout,
                input_shape=(192, 352, 1), num_classes=5):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs

    for i in range(num_blocks):
        filters = base_filters * (2 ** i)
        drop_i  = min(dropout + i * 0.05, 0.4)

        residual = x

        x = layers.Conv2D(filters, (3, 3), padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)

        x = layers.Conv2D(filters, (3, 3), padding='same')(x)
        x = layers.BatchNormalization()(x)

        if residual.shape[-1] != filters:
            residual = layers.Conv2D(filters, (1, 1), padding='same')(residual)
            residual = layers.BatchNormalization()(residual)

        x = layers.Add()([x, residual])
        x = layers.ReLU()(x)

        x = layers.MaxPooling2D((2, 2))(x)
        x = layers.Dropout(drop_i)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(dense_units, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(num_classes, activation='softmax')(x)

    return tf.keras.Model(inputs=inputs, outputs=x, name='CNN_B')

### **Monitoreo de Recursos y Logging**

`ResourceMonitor` registra al final de cada época el consumo máximo de RAM, el consumo máximo de VRAM y la utilización promedio de GPU. `log_training` guarda una fila por modelo en `CNN-B_training.csv` y `log_results` guarda las métricas de test en `CNN-B_results.csv`.

In [10]:
# ── Utilidades de logging ───────────────────────────────────────────────────────
def _append_csv(filepath, row):
    filepath  = Path(filepath)
    write_hdr = not filepath.exists()
    with open(filepath, 'a', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=row.keys())
        if write_hdr:
            w.writeheader()
        w.writerow(row)

# ── Monitor de recursos ──────────────────────────────────────────────────────────
try:
    pynvml.nvmlInit()
    _GPU_HANDLE    = pynvml.nvmlDeviceGetHandleByIndex(0)
    _GPU_AVAILABLE = True
except Exception:
    _GPU_AVAILABLE = False

class ResourceMonitor(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.ram_max   = 0.0
        self.vram_max  = 0.0
        self.gpu_utils = []
        self._pid      = os.getpid()

    def on_epoch_end(self, epoch, logs=None):
        self.ram_max = max(self.ram_max, psutil.Process().memory_info().rss / 1e9)
        if _GPU_AVAILABLE:
            for proc in pynvml.nvmlDeviceGetComputeRunningProcesses(_GPU_HANDLE):
                if proc.pid == self._pid:
                    self.vram_max = max(self.vram_max, proc.usedGpuMemory / 1e9)
                    break
            try:
                for u in pynvml.nvmlDeviceGetProcessUtilization(_GPU_HANDLE, 1000000):
                    if u.pid == self._pid:
                        self.gpu_utils.append(u.smUtil)
                        break
            except Exception:
                self.gpu_utils.append(pynvml.nvmlDeviceGetUtilizationRates(_GPU_HANDLE).gpu)


def log_training(cfg, history, monitor, t_start, t_end, model_path, history_path):
    row = {
        'run_id':         cfg['run_id'],
        'fecha':          datetime.now().strftime('%d/%m/%Y'),
        'arquitectura':   ARQUITECTURA,
        'dataset':        DATASET,
        'img_width':      IMG_WIDTH,
        'img_height':     IMG_HEIGHT,
        'batch_size':     BATCH_SIZE,
        'seed':           SEED,
        'optuna_run_id':  f"{OPTUNA_RUN_ID}_trial{cfg['optuna_trial']}",
        'num_blocks':     cfg['num_blocks'],
        'base_filters':   cfg['base_filters'],
        'ft_blocks':      None,
        'dense_units':    DENSE_UNITS,
        'dropout':        round(cfg['dropout'], 4),
        'learning_rate':  cfg['learning_rate'],
        'optimizer':      OPTIMIZER,
        'loss_function':  LOSS_FUNCTION,
        'es_patience':    ES_PATIENCE,
        'es_min_delta':   ES_MIN_DELTA,
        'rlr_factor':     RLR_FACTOR,
        'rlr_patience':   RLR_PATIENCE,
        'rlr_min_delta':  RLR_MIN_DELTA,
        'rlr_min_lr':     RLR_MIN_LR,
        'epochs_max':     EPOCH_MAX,
        'epochs_trained': len(history.history['val_loss']),
        't_start':        datetime.fromtimestamp(t_start).strftime('%d/%m - %H:%M:%S'),
        't_end':          datetime.fromtimestamp(t_end).strftime('%d/%m - %H:%M:%S'),
        't_training':     round((t_end - t_start) / 60, 2),
        'ram_max_gb':     round(monitor.ram_max, 2),
        'vram_max_gb':    round(monitor.vram_max, 2),
        'gpu_avg':        round(float(np.mean(monitor.gpu_utils)), 1) if monitor.gpu_utils else None,
        'model_path':     str(model_path.relative_to(REPO_ROOT)),
        'history_path':   str(history_path.relative_to(REPO_ROOT)),
    }
    _append_csv(MODELS_PATH / 'CNN-B_training.csv', row)
    print(f"  CNN-B_training.csv actualizado — {cfg['run_id']}")


def log_results(run_id, y_true, y_pred, history, test_loss, test_acc, classes, notas=''):
    best_epoch  = int(np.argmin(history['val_loss']))
    report_dict = classification_report(y_true, y_pred, target_names=classes, output_dict=True)
    row = {
        'run_id':        run_id,
        'best_epoch':    best_epoch + 1,
        'best_val_loss': round(history['val_loss'][best_epoch], 4),
        'best_val_acc':  round(history['val_accuracy'][best_epoch], 4),
        'test_loss':     round(test_loss, 4),
        'test_acc':      round(test_acc, 4),
        'balanced_acc':  round(balanced_accuracy_score(y_true, y_pred), 4),
        'macro_f1':      round(report_dict['macro avg']['f1-score'], 4),
        'weighted_f1':   round(report_dict['weighted avg']['f1-score'], 4),
    }
    for cls in classes:
        row[f'precision_{cls}'] = round(report_dict[cls]['precision'], 4)
        row[f'recall_{cls}']    = round(report_dict[cls]['recall'],    4)
        row[f'f1_{cls}']        = round(report_dict[cls]['f1-score'],  4)
        row[f'support_{cls}']   = int(report_dict[cls]['support'])
    row['notas'] = notas
    _append_csv(MODELS_PATH / 'CNN-B_results.csv', row)
    print(f'  CNN-B_results.csv actualizado — {run_id}')
    return row

### **Entrenamiento**

Para cada una de las 3 configuraciones se construye y entrena el modelo desde cero, con la misma semilla (`SEED=42`) para todas. Se utilizan los mismos callbacks que en la búsqueda Optuna: `EarlyStopping` (con restauración de los mejores pesos), `ReduceLROnPlateau`, `ModelCheckpoint` (guarda el mejor modelo según `val_loss`) y `ResourceMonitor`.

In [11]:
# ── ENTRENAMIENTO ─────────────────────────────────────────────────────────────
HISTORIES = {}

for cfg in CONFIGS:
    run_id = cfg['run_id']
    print(f'\n{"="*60}\nEntrenando {run_id} (trial Optuna #{cfg["optuna_trial"]})\n{"="*60}')

    tf.keras.backend.clear_session()
    gc.collect()
    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)

    train_data = make_dataset(DATA_PATH / 'train', shuffle=True)
    val_data   = make_dataset(DATA_PATH / 'val',   shuffle=False)

    model = build_cnn_b(
        num_blocks   = cfg['num_blocks'],
        base_filters = cfg['base_filters'],
        dense_units  = DENSE_UNITS,
        dropout      = cfg['dropout'],
        input_shape  = (IMG_HEIGHT, IMG_WIDTH, CHANNELS),
        num_classes  = NUM_CLASSES,
    )
    model.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate=cfg['learning_rate']),
        loss      = LOSS_FUNCTION,
        metrics   = ['accuracy'],
    )

    checkpoint_path  = MODELS_PATH / f'{run_id}.keras'
    history_path     = MODELS_PATH / f'{run_id}_history.json'
    resource_monitor = ResourceMonitor()

    callbacks = [
        EarlyStopping(
            monitor              = 'val_loss',
            patience             = ES_PATIENCE,
            min_delta            = ES_MIN_DELTA,
            restore_best_weights = True,
        ),
        ReduceLROnPlateau(
            monitor   = 'val_loss',
            factor    = RLR_FACTOR,
            patience  = RLR_PATIENCE,
            min_delta = RLR_MIN_DELTA,
            min_lr    = RLR_MIN_LR,
        ),
        ModelCheckpoint(
            filepath       = checkpoint_path,
            monitor        = 'val_loss',
            save_best_only = True,
            verbose        = 1,
        ),
        resource_monitor,
    ]

    t_start = time.time()
    history = model.fit(
        train_data,
        validation_data = val_data,
        epochs          = EPOCH_MAX,
        callbacks        = callbacks,
    )
    t_end = time.time()

    with open(history_path, 'w') as f:
        json.dump(history.history, f)

    log_training(cfg, history, resource_monitor, t_start, t_end, checkpoint_path, history_path)
    HISTORIES[run_id] = history.history


Entrenando CNN-B-1 (trial Optuna #29)
Found 46497 files belonging to 5 classes.
Found 11624 files belonging to 5 classes.
Epoch 1/100
1453/1454 ━━━━━━━━━━━━━━━━━━━━ 0s 241ms/step - accuracy: 0.4825 - loss: 1.3540
Epoch 1: val_loss improved from None to 1.95902, saving model to /home/vorsi/pfc/TESIS_bioingenieria_JonquieresOrsi/Códigos/Modelos/Fase 1/CNN-B/CNN-B-1.keras

Epoch 1: finished saving model to /home/vorsi/pfc/TESIS_bioingenieria_JonquieresOrsi/Códigos/Modelos/Fase 1/CNN-B/CNN-B-1.keras
1454/1454 ━━━━━━━━━━━━━━━━━━━━ 372s 252ms/step - accuracy: 0.5732 - loss: 1.1507 - val_accuracy: 0.5095 - val_loss: 1.9590 - learning_rate: 1.1978e-04
Epoch 2/100
1453/1454 ━━━━━━━━━━━━━━━━━━━━ 0s 242ms/step - accuracy: 0.6574 - loss: 0.9597
Epoch 2: val_loss did not improve from 1.95902
1454/1454 ━━━━━━━━━━━━━━━━━━━━ 366s 252ms/step - accuracy: 0.6675 - loss: 0.9321 - val_accuracy: 0.4974 - val_loss: 3.2347 - learning_rate: 1.1978e-04
Epoch 3/100
1453/1454 ━━━━━━━━━━━━━━━━━━━━ 0s 242ms/step -

### **Resultados**

#### **Reconstrucción de contexto tras reinicio de kernel**

Correr la siguiente celda **solo si el kernel fue reiniciado** después del entrenamiento. Antes de correrla, ejecutar todas las celdas anteriores al bloque de entrenamiento (imports, GPU, paths, configuración, carga de datos, arquitectura, logging) para que `CONFIGS` y `test_data` estén disponibles.

In [9]:
# ── RECONSTRUCCIÓN DE CONTEXTO ────────────────────────────────────────────────

# Histories desde JSON
HISTORIES = {}
for cfg in CONFIGS:
    with open(MODELS_PATH / f"{cfg['run_id']}_history.json") as f:
        HISTORIES[cfg['run_id']] = json.load(f)

# y_true + predicciones desde los .keras guardados (no re-escribe CSV)
y_true = np.concatenate([y for x, y in test_data], axis=0)
EVAL = {}
for cfg in CONFIGS:
    run_id = cfg['run_id']
    model  = tf.keras.models.load_model(MODELS_PATH / f'{run_id}.keras')

    y_pred              = np.argmax(model.predict(test_data, verbose=0), axis=1)
    test_loss, test_acc = model.evaluate(test_data, verbose=0)

    best_epoch  = int(np.argmin(HISTORIES[run_id]['val_loss']))
    report_dict = classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True)

    EVAL[run_id] = {
        'y_pred': y_pred,
        'report': report_dict,
        'report_row': {
            'run_id':        run_id,
            'best_epoch':    best_epoch + 1,
            'best_val_loss': round(HISTORIES[run_id]['val_loss'][best_epoch], 4),
            'best_val_acc':  round(HISTORIES[run_id]['val_accuracy'][best_epoch], 4),
            'test_loss':     round(test_loss, 4),
            'test_acc':      round(test_acc, 4),
            'balanced_acc':  round(balanced_accuracy_score(y_true, y_pred), 4),
            'macro_f1':      round(report_dict['macro avg']['f1-score'], 4),
            'weighted_f1':   round(report_dict['weighted avg']['f1-score'], 4),
            **{f'precision_{cls}': round(report_dict[cls]['precision'], 4) for cls in CLASSES},
            **{f'recall_{cls}':    round(report_dict[cls]['recall'],    4) for cls in CLASSES},
            **{f'f1_{cls}':        round(report_dict[cls]['f1-score'],  4) for cls in CLASSES},
            **{f'support_{cls}':   int(report_dict[cls]['support'])        for cls in CLASSES},
        },
    }

    print(f"{run_id}: test_loss={test_loss:.4f} | test_acc={test_acc:.4f} | balanced_acc={EVAL[run_id]['report_row']['balanced_acc']:.4f}")
    del model
    gc.collect()

print('\nContexto reconstruido.')

KeyboardInterrupt: 

#### **Curvas de Entrenamiento**

In [ ]:
import json
HISTORIES = {}
for cfg in CONFIGS:
    with open(MODELS_PATH / f"{cfg['run_id']}_history.json") as f:
        HISTORIES[cfg['run_id']] = json.load(f)

In [ ]:
TRAIN_COLOR = '#061F4A'
VAL_COLOR   = '#B11919'

fig, axes = plt.subplots(len(CONFIGS), 2, figsize=(12, 4 * len(CONFIGS)))

for i, cfg in enumerate(CONFIGS):
    run_id = cfg['run_id']
    h = HISTORIES[run_id]

    # Loss
    axes[i, 0].plot(h['loss'],     color=TRAIN_COLOR, label='Train')
    axes[i, 0].plot(h['val_loss'], color=VAL_COLOR,   label='Val')
    axes[i, 0].set_title(run_id)
    axes[i, 0].set_xlabel('Época')
    axes[i, 0].set_ylabel('Loss')
    axes[i, 0].set_ylim(0, 4)
    axes[i, 0].legend()
    axes[i, 0].grid(True, alpha=0.3)

    # Accuracy
    axes[i, 1].plot(h['accuracy'],     color=TRAIN_COLOR, label='Train')
    axes[i, 1].plot(h['val_accuracy'], color=VAL_COLOR,   label='Val')
    axes[i, 1].set_title(run_id)
    axes[i, 1].set_xlabel('Época')
    axes[i, 1].set_ylabel('Accuracy')
    axes[i, 1].set_ylim(0, 1)
    axes[i, 1].legend()
    axes[i, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#### **Evaluación en Test**

In [14]:
# ── Evaluación en test ────────────────────────────────────────────────────────
y_true = np.concatenate([y for x, y in test_data], axis=0)

EVAL = {}
for cfg in CONFIGS:
    run_id = cfg['run_id']
    model  = tf.keras.models.load_model(MODELS_PATH / f'{run_id}.keras')

    y_pred = np.argmax(model.predict(test_data, verbose=0), axis=1)
    test_loss, test_acc = model.evaluate(test_data, verbose=0)

    report_row = log_results(run_id, y_true, y_pred, HISTORIES[run_id], test_loss, test_acc, CLASSES)
    EVAL[run_id] = {
        'y_pred':     y_pred,
        'report':     classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True),
        'report_row': report_row,
    }
    del model
    gc.collect()

  CNN-B_results.csv actualizado — CNN-B-1
  CNN-B_results.csv actualizado — CNN-B-2
  CNN-B_results.csv actualizado — CNN-B-3


#### **Matrices de Confusión**

In [ ]:
fig, axes = plt.subplots(1, len(CONFIGS), figsize=(6 * len(CONFIGS), 5))

for i, cfg in enumerate(CONFIGS):
    run_id  = cfg['run_id']
    cm_norm = confusion_matrix(y_true, EVAL[run_id]['y_pred'], normalize='true') * 100
    sns.heatmap(cm_norm, annot=True, fmt='.1f', cmap='Reds',
                xticklabels=CLASSES, yticklabels=CLASSES,
                vmin=0, vmax=100, ax=axes[i], cbar=False)
    axes[i].set_xlabel('Predicho')
    axes[i].set_ylabel('Real')
    axes[i].set_title(run_id.rsplit('-', 1)[0])

plt.tight_layout()
plt.show()

#### **Tablas de Métricas**

In [16]:
for cfg in CONFIGS:
    run_id    = cfg['run_id']
    df_report = pd.DataFrame(EVAL[run_id]['report']).transpose()
    print(f'\n{run_id}')
    display(df_report.style.format({
        'precision': '{:.3f}',
        'recall':    '{:.3f}',
        'f1-score':  '{:.3f}',
        'support':   '{:.0f}',
    }))


CNN-B-1


,precision,recall,f1-score,support
CD,0.872,0.774,0.820,1814
HYP,0.698,0.540,0.609,572
MI,0.745,0.645,0.692,1006
NORM,0.822,0.897,0.858,3954
STTC,0.768,0.796,0.782,2912
accuracy,0.802,0.802,0.802,1
macro avg,0.781,0.730,0.752,10258
weighted avg,0.801,0.802,0.799,10258



CNN-B-2


,precision,recall,f1-score,support
CD,0.801,0.847,0.823,1814
HYP,0.648,0.647,0.647,572
MI,0.721,0.682,0.701,1006
NORM,0.861,0.852,0.857,3954
STTC,0.796,0.794,0.795,2912
accuracy,0.807,0.807,0.807,1
macro avg,0.765,0.764,0.765,10258
weighted avg,0.806,0.807,0.806,10258



CNN-B-3


,precision,recall,f1-score,support
CD,0.802,0.818,0.809,1814
HYP,0.696,0.488,0.573,572
MI,0.661,0.698,0.679,1006
NORM,0.831,0.868,0.849,3954
STTC,0.775,0.748,0.761,2912
accuracy,0.787,0.787,0.787,1
macro avg,0.753,0.724,0.734,10258
weighted avg,0.785,0.787,0.785,10258


In [17]:
# ── Tabla comparativa ────────────────────────────────────────────────────────
comp_rows = []
for cfg in CONFIGS:
    run_id = cfg['run_id']
    row    = EVAL[run_id]['report_row']
    comp = {'run_id': run_id, 'balanced_acc': row['balanced_acc'], 'macro_f1': row['macro_f1']}
    for cls in CLASSES:
        comp[f'f1_{cls}'] = row[f'f1_{cls}']
    comp_rows.append(comp)

df_comp = pd.DataFrame(comp_rows).set_index('run_id')
display(df_comp.style.format('{:.4f}'))

,balanced_acc,macro_f1,f1_CD,f1_HYP,f1_MI,f1_NORM,f1_STTC
run_id,,,,,,,
CNN-B-1,0.7304,0.7519,0.8199,0.6089,0.6915,0.8576,0.7815
CNN-B-2,0.7644,0.7647,0.8235,0.6474,0.7011,0.8566,0.7949
CNN-B-3,0.7239,0.7344,0.8095,0.5735,0.6789,0.8489,0.7612


In [ ]:
import IPython

# ── Liberar memoria ───────────────────────────────────────────────────────────
print('Reiniciando kernel...')
IPython.Application.instance().kernel.do_shutdown(restart=True)